<div style="width: 30em; float: right; padding: 3em; border: 5px red solid; background-color: darkred; color: white"><p style="font-size: large; font-weight:bold">Rename this notebook before running any cells!</p><ol><li>Right-click on the tab title above or on the notebook in the file-browser on the left and select rename.</li><li>Remove the "_orig" part of the file name.</li></ol><p style="font-size: small">Your edits to a file ending in <code>_orig</code> may be overwritten the next time the course materials are updated.</p></div>

# Anatomy of Python Functions

**IFI8410 &mdash; Session 4: Functions and Decomposition**

The companion notebook, *Deep Dive: Why Functions Are Designed This Way*, argues about
**why** to write functions a certain way. This notebook is about the **machinery**: every
part of a function, what each part is called, and what Python actually does with it.

Run every cell. Several of them are designed to fail, or to produce a surprising result
&mdash; those are the ones worth slowing down for.

### Contents

| Section | Topic |
|---|---|
| 1 | The anatomy of a `def` statement |
| 2 | Scope: where names live (LEGB) |
| 3 | Parameters and arguments |
| 4 | Named (keyword) arguments and defaults |
| 5 | Flexible signatures: `*args`, `**kwargs`, `/` and `*` |
| 6 | Return values |
| 7 | Type annotations |
| 8 | `lambda`: functions as expressions |
| 9 | Functions are objects |
| 10 | Closures and currying |

### The dataset

Where an example needs data, it uses the coffee-cart table from Session 3 &mdash; a **list
of dictionaries**, one dictionary per row, dictionary keys as column names.

In [ ]:
# The Session 3 coffee-cart table. Small enough to check answers by eye.

sales = [
    {"id": 101, "item": "coffee", "category": "drink", "price": 3.50, "day": "Mon", "customer_type": "student"},
    {"id": 102, "item": "tea",    "category": "drink", "price": 2.75, "day": "Mon", "customer_type": "faculty"},
    {"id": 103, "item": "muffin", "category": "food",  "price": 2.50, "day": "Mon", "customer_type": "student"},
    {"id": 104, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "customer_type": "student"},
    {"id": 105, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Tue", "customer_type": "staff"},
    {"id": 106, "item": "coffee", "category": "drink", "price": 3.50, "day": "Wed", "customer_type": "faculty"},
    {"id": 107, "item": "cookie", "category": "food",  "price": 1.75, "day": "Wed", "customer_type": "student"},
]

print("rows:", len(sales))
print("columns:", list(sales[0]))

---

## 1. The Anatomy of a `def` Statement

Every part of a function definition has a name. Learning the vocabulary makes error
messages and documentation readable.

```text
  def  discounted_price( price: float, discount: float = 0.0 ) -> float:
  ---  ----------------  --------------------------------      -----
   |          |                        |                          |
keyword     name                  parameter list            return annotation
                                                                          
      """Return price after discount."""     <- docstring
      if not 0 <= discount <= 1:             <- body
          raise ValueError(...)                
      return price * (1 - discount)          <- return statement
```

| Part | Required? | What it does |
|---|---|---|
| `def` | yes | The keyword that begins a function definition. |
| **name** | yes | The name the function is bound to. Follows variable naming rules. |
| **parameter list** | parentheses required, contents optional | Names the inputs. |
| **return annotation** (`-> float`) | no | Documents the result type. |
| **docstring** | no, but expected | The contract, as text. |
| **body** | yes | The indented block that runs on each call. |
| **`return`** | no | Sends a value back. Without it, the function returns `None`. |

In [ ]:
def discounted_price(price: float, discount: float = 0.0) -> float:
    """Return price after applying a fractional discount.

    Args:
        price: The undiscounted price.
        discount: A fraction from 0.0 through 1.0. Defaults to no discount.

    Returns:
        The discounted price.
    """
    if not 0.0 <= discount <= 1.0:
        raise ValueError("discount must be between 0.0 and 1.0")
    return price * (1 - discount)


print(discounted_price(3.50))
print(discounted_price(3.50, 0.10))

### Definition time versus call time

A `def` statement is **executed**, not just read. Running it creates a function object and
binds it to a name. The body does **not** run until the function is called.

In [ ]:
print("before the def")

def greet(name):
    print(f"   ...the body runs now, for {name}")
    return f"Hello, {name}"

print("after the def -- the body has NOT run yet")
print("the name 'greet' is now bound to:", greet)

message = greet("Amina")      # NOW the body runs
print("returned:", message)

This is why a typo inside a function body is not reported until you call it. Python checks
*syntax* at definition time, but looks up names at *call* time.

In [ ]:
def uses_a_name_that_does_not_exist():
    return undefined_name + 1      # Python accepts this definition

print("defined without complaint")

try:
    uses_a_name_that_does_not_exist()
except NameError as error:
    print(f"but calling it: NameError: {error}")

In [ ]:
### Try it: write a function `cart_total(prices, tax_rate)` with
### a docstring, a return annotation, and a `return` statement.
### Then call it and print the result.

### Enter your code here ###

---

## 2. Scope: Where Names Live

When Python meets a name, it searches four scopes **in order**. The rule is called
**LEGB**:

| Scope | Meaning | Example |
|---|---|---|
| **L** ocal | Inside the current function call | a parameter, or a name assigned in the body |
| **E** nclosing | Inside an outer function, for a nested function | see Section 10 |
| **G** lobal | At the top level of the module / notebook | `sales` above |
| **B** uilt-in | Always available | `len`, `print`, `sum` |

The first match wins, and the search **stops there**.

In [ ]:
message = "global"        # G

def outer():
    message = "enclosing"  # E

    def inner():
        message = "local"  # L
        print("inner sees:", message)

    inner()
    print("outer sees:", message)

outer()
print("module sees:", message)

Each layer found its own `message` first. Now remove the local one and watch the search
continue outward:

In [ ]:
message = "global"

def outer():
    message = "enclosing"

    def inner():
        print("inner sees:", message)   # no local -> found in ENCLOSING

    inner()

outer()


def standalone():
    print("standalone sees:", message)  # no local, no enclosing -> found in GLOBAL

standalone()


def uses_builtin():
    print("len is the BUILT-IN:", len([1, 2, 3]))

uses_builtin()

### Assignment creates a local name

This is the rule that surprises people most. **Assigning to a name anywhere in a function
body makes that name local for the whole body** &mdash; even on lines before the assignment.

In [ ]:
count = 10

def broken():
    print(count)     # Python already decided `count` is local, because of the next line
    count = 5

try:
    broken()
except UnboundLocalError as error:
    print(f"UnboundLocalError: {error}")

Reading is fine; it is the *assignment* that changes the classification:

In [ ]:
count = 10

def reads_only():
    print("reads the global just fine:", count)

reads_only()

### `global` and `nonlocal`

Two keywords override the default. Both are worth **recognizing** and usually worth
**avoiding** &mdash; the deep-dive notebook explains why an explicit parameter is better.

- `global name` &mdash; assignment inside the function rebinds the *module-level* name.
- `nonlocal name` &mdash; assignment rebinds the nearest *enclosing function's* name.

In [ ]:
total = 0

def add_global(value):
    global total
    total += value

add_global(5)
add_global(7)
print("global total:", total)


def make_accumulator():
    running = 0

    def add(value):
        nonlocal running        # without this: UnboundLocalError
        running += value
        return running

    return add

accumulate = make_accumulator()
print(accumulate(5), accumulate(7), accumulate(3))

That second example is a **closure**, and it is the foundation of Section 10.

### Local names disappear

Names created in a call vanish when the call ends. Nothing leaks into the notebook.

In [ ]:
def compute():
    scratch = "temporary"
    return len(scratch)

print(compute())

try:
    print(scratch)
except NameError as error:
    print(f"NameError: {error}")

In [ ]:
### Try it: predict the output BEFORE running, then run it.
### Which `price` does each print statement find?

price = 100

def show():
    price = 50
    print("A:", price)

show()
print("B:", price)

### Now add the line `global price` as the first line of show() and re-run.
### Enter your code here ###

---

## 3. Parameters and Arguments

Two words that are often confused, and the distinction makes documentation readable:

- A **parameter** is the name in the `def` line. It is part of the function.
- An **argument** is the value you pass at the call. It belongs to the call.

```python
def add(a, b):        # a and b are PARAMETERS
    return a + b

add(2, 3)             # 2 and 3 are ARGUMENTS
```

### Positional arguments

By default, arguments are matched to parameters **by position**, left to right.

In [ ]:
def describe_sale(item, price, day):
    return f"{item} at ${price:.2f} on {day}"

print(describe_sale("coffee", 3.50, "Mon"))

# Position is all that matters -- so the wrong ORDER is not an error, just wrong:
print(describe_sale("Mon", 3.50, "coffee"))

Note that the second call produced nonsense without raising anything. Python matched three
values to three parameters exactly as instructed. This is a strong argument for the
**named arguments** in Section 4.

Passing the wrong **number** of arguments *is* an error, and the message names the
parameters:

In [ ]:
for bad_call in [
    lambda: describe_sale("coffee", 3.50),                    # too few
    lambda: describe_sale("coffee", 3.50, "Mon", "student"),  # too many
]:
    try:
        bad_call()
    except TypeError as error:
        print(f"TypeError: {error}")

### Arguments are passed by assignment

Python does not copy arguments. The parameter name is **bound to the same object** the
caller passed. The consequence depends on whether that object can be changed in place.

In [ ]:
def rebind(values):
    values = ["a", "b"]        # rebinds the LOCAL name only
    return values

def mutate(values):
    values.append("a")         # changes the OBJECT the caller still holds
    return values

original = [1, 2]
rebind(original)
print("after rebind:", original)    # unchanged

mutate(original)
print("after mutate:", original)    # changed!

The rule: **rebinding a parameter is invisible to the caller; mutating the object it points
to is not.** Section 7 of the deep-dive notebook explores what that means for your
function contracts.

In [ ]:
### Try it: write `busiest_day(rows)` that takes the `sales` table and returns
### the day with the most transactions. Use a parameter -- do not read the
### global `sales` from inside the function.

### Enter your code here ###

---

## 4. Named (Keyword) Arguments and Defaults

You can pass an argument **by name** instead of by position. These are called **keyword
arguments** or **named arguments**.

In [ ]:
def describe_sale(item, price, day):
    return f"{item} at ${price:.2f} on {day}"

# All four calls are equivalent:
print(describe_sale("coffee", 3.50, "Mon"))
print(describe_sale("coffee", 3.50, day="Mon"))
print(describe_sale("coffee", price=3.50, day="Mon"))
print(describe_sale(day="Mon", price=3.50, item="coffee"))   # order is free

Two rules govern them:

1. **Keyword arguments may be given in any order**, because the name does the matching.
2. **Positional arguments must come first.** A positional argument after a keyword
   argument is a syntax error.

In [ ]:
# `compile` parses source without running it -- ideal for showing a syntax rule.
try:
    compile('describe_sale(item="coffee", 3.50, "Mon")', "<demo>", "eval")
except SyntaxError as error:
    print(f"SyntaxError: {error.msg}")

### Why named arguments matter

Compare these two calls. Only one of them can be read without opening the documentation.

```python
create_report(data, True, False, True)              # what are these?
create_report(data, header=True, index=False, totals=True)
```

A good rule: **if an argument is a bare `True`, `False`, or a number whose meaning is not
obvious from the value, pass it by name.**

### Default values

A parameter with a default becomes **optional**.

In [ ]:
def summarize(rows, group_by="day", round_to=2):
    """Return {group value: total price}, rounded."""
    totals = {}
    for row in rows:
        key = row[group_by]
        totals[key] = round(totals.get(key, 0) + row["price"], round_to)
    return totals

print(summarize(sales))                          # both defaults
print(summarize(sales, "category"))              # positional override
print(summarize(sales, group_by="customer_type"))  # named override -- clearer
print(summarize(sales, round_to=0))              # skip group_by, set only round_to

That last call is the real payoff of defaults: you can set a *later* parameter without
restating the earlier ones.

**Parameters with defaults must come after parameters without them**, otherwise a call
could not tell which value went where.

In [ ]:
try:
    compile("def f(a=1, b): pass", "<demo>", "exec")
except SyntaxError as error:
    print(f"SyntaxError: {error.msg}")

### The mutable default trap

A default value is evaluated **once**, when the `def` statement runs &mdash; not on each
call. A mutable default is therefore shared by every call that omits it.

In [ ]:
def collect_broken(item, bucket=[]):        # DO NOT DO THIS
    bucket.append(item)
    return bucket

print(collect_broken("a"))
print(collect_broken("b"))    # ['a', 'b'] -- last call's data is still there
print(collect_broken.__defaults__)   # the ONE shared list, visible on the function


def collect(item, bucket=None):             # the correct idiom
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(collect("a"))
print(collect("b"))           # ['b'] -- a fresh list every call

In [ ]:
### Try it: give `describe_sale` a `currency` parameter defaulting to "$",
### then call it once with the default and once with "€" passed by name.

### Enter your code here ###

---

## 5. Flexible Signatures: `*args`, `**kwargs`, `/` and `*`

### `*args` collects extra positional arguments into a tuple

In [ ]:
def total(*prices):
    """Accept any number of positional prices."""
    print("  received a", type(prices).__name__, ":", prices)
    return sum(prices)

print(total(3.50, 2.75))
print(total(3.50, 2.75, 2.50, 1.75))
print(total())        # zero arguments is fine -> empty tuple

### `**kwargs` collects extra keyword arguments into a dictionary

In [ ]:
def make_row(**fields):
    """Accept any number of named fields."""
    print("  received a", type(fields).__name__, ":", fields)
    return fields

print(make_row(item="latte", price=4.25, day="Thu"))

### The full ordering

A signature may combine all of these, and the order is fixed:

```text
def f(positional_only, /, standard, *args, keyword_only, **kwargs):
```

- Everything **before `/`** can *only* be passed positionally.
- Everything **after `*`** (or after `*args`) can *only* be passed by name.

In [ ]:
def report(title, /, rows, *extra, verbose=False, **options):
    print("title       :", title)
    print("rows        :", rows)
    print("extra       :", extra)
    print("verbose     :", verbose)
    print("options     :", options)

report("Weekly", 7, "a", "b", verbose=True, color="red", width=80)

In [ ]:
# `title` is positional-only: passing it by name fails.
try:
    report(title="Weekly", rows=7)
except TypeError as error:
    print(f"TypeError: {error}")

# `verbose` is keyword-only: passing it positionally fails.
try:
    report("Weekly", 7, True)     # True lands in *extra, not in verbose
except TypeError as error:
    print(f"TypeError: {error}")
else:
    print("(no error -- True was swallowed by *extra; that is the trap)")

**Why keyword-only parameters are useful:** they make a confusing call *impossible*. If
`verbose` can only be passed as `verbose=True`, nobody can ever write `report("Weekly", 7,
True)` and wonder what the `True` means.

### Unpacking at the call site

The same `*` and `**` symbols work in reverse: they **spread** a collection into arguments.

In [ ]:
def describe_sale(item, price, day):
    return f"{item} at ${price:.2f} on {day}"

values = ["coffee", 3.50, "Mon"]
print(describe_sale(*values))          # spread a list into positional arguments

row = {"item": "muffin", "price": 2.50, "day": "Tue"}
print(describe_sale(**row))            # spread a dict into keyword arguments

# This is why matching key names to parameter names is convenient:
print(describe_sale(**{k: sales[0][k] for k in ("item", "price", "day")}))

In [ ]:
### Try it: write `log(message, *tags, level="INFO", **context)` that prints
### all four parts. Call it with two tags, a non-default level, and one
### extra keyword.

### Enter your code here ###

---

## 6. Return Values

### `return` ends the call immediately

In [ ]:
def classify(price):
    if price < 2.00:
        return "cheap"
    if price < 3.25:
        return "moderate"
    return "expensive"
    print("this line can never run")     # unreachable

for price in [1.75, 2.75, 3.50]:
    print(price, "->", classify(price))

### Every function returns something

A function with no `return`, or a bare `return`, returns `None`.

In [ ]:
def no_return():
    pass

def bare_return():
    return

def explicit_none():
    return None

print(no_return(), bare_return(), explicit_none())
print(all(f() is None for f in (no_return, bare_return, explicit_none)))

### Returning several values

Python has no "multiple return values". It returns **one tuple**, and the call site
**unpacks** it. That distinction explains everything about the behavior.

In [ ]:
def price_range(rows):
    """Return the lowest and highest price in the table."""
    prices = [row["price"] for row in rows]
    return min(prices), max(prices)      # parentheses are optional -- it is a tuple

result = price_range(sales)
print("as a tuple :", result, type(result).__name__)

low, high = price_range(sales)           # unpacking
print("unpacked   :", low, high)

low, _ = price_range(sales)              # _ is the convention for "not needed"
print("just the low:", low)

For more than two or three values, a **dictionary** makes the result self-describing &mdash;
the caller reads a name instead of counting positions.

In [ ]:
def summary_stats(rows):
    """Return named summary statistics, so callers never count tuple positions."""
    prices = [row["price"] for row in rows]
    return {
        "count": len(prices),
        "lowest": min(prices),
        "highest": max(prices),
        "mean": sum(prices) / len(prices),
    }

stats = summary_stats(sales)
print(stats)
print("mean only:", stats["mean"])       # readable at the call site

### Return a consistent type

A function that sometimes returns a number and sometimes a string forces every caller to
check. Decide one type and stick to it &mdash; or return `None` for "no answer" and say so
in the docstring.

In [ ]:
def mean_price_bad(rows):
    if not rows:
        return "no data"       # a string...
    return sum(r["price"] for r in rows) / len(rows)   # ...or a float

def mean_price(rows):
    """Return the mean price, or None when there are no rows."""
    if not rows:
        return None
    return sum(r["price"] for r in rows) / len(rows)

print(mean_price_bad([]), "|", mean_price([]))

# The bad version breaks the moment a caller does arithmetic:
try:
    print(mean_price_bad([]) * 2)
except TypeError as error:
    print(f"TypeError: {error}")
else:
    print("(no error -- 'no datano data', which is worse than an error)")

In [ ]:
### Try it: write `cheapest_item(rows)` returning a (item, price) tuple,
### then unpack it at the call site into two variables.

### Enter your code here ###

---

## 7. Type Annotations

An annotation records the **expected** type of a parameter or the return value.

```python
def mean_price(rows: list[dict]) -> float:
    ...
```

Read it as: *`rows` is expected to be a list of dictionaries; the result is expected to be
a float.*

### Annotations are not enforced

This is the single most important fact about them. Python stores annotations and otherwise
ignores them at runtime.

In [ ]:
def double(value: int) -> int:
    return value * 2

print(double(5))          # as annotated
print(double("ha"))       # a string -- runs anyway, returns 'haha'
print(double([1, 2]))     # a list -- runs anyway
print(double.__annotations__)

So what are they *for*?

| Audience | What it does with the annotation |
|---|---|
| **The next reader** | Answers "what do I pass?" without reading the body. |
| **Your editor** | Autocompletion and inline warnings. |
| **A type checker** (mypy, pyright) | Reports likely mismatches *before* you run. |
| **The Python runtime** | Nothing. |

Because the runtime ignores them, annotations should be paired with **validation** at any
boundary where bad input would be costly &mdash; see Section 6 of the deep-dive notebook.

### The vocabulary

In [ ]:
# Built-in types are used directly as annotations.
def f1(name: str, count: int, price: float, ok: bool) -> str:
    return f"{name} {count} {price} {ok}"

# Containers say what they hold (Python 3.9+).
def f2(prices: list[float], row: dict[str, object], pair: tuple[int, str]) -> set[str]:
    return {str(len(prices)), str(len(row)), str(pair[1])}

print(f1.__annotations__)
print(f2.__annotations__)

In [ ]:
# "Either this or that" uses the | operator (Python 3.10+).
def find_price(rows: list[dict], item: str) -> float | None:
    """Return the item's price, or None if the item is not in the table."""
    for row in rows:
        if row["item"] == item:
            return row["price"]
    return None

print(find_price(sales, "coffee"))
print(find_price(sales, "sandwich"))     # None -- and the annotation warned you

`float | None` is the annotation you will write most often in data work. It is the honest
way to say *"this may have no answer"*, and it tells the caller to check before doing
arithmetic.

A few more you will meet:

In [ ]:
from typing import Any, Callable, Optional

# Optional[X] is an older spelling of X | None -- you will see both.
def older_style(item: str) -> Optional[float]:
    return None

# Callable[[argument types], return type] annotates a FUNCTION parameter.
def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))

# Any means "deliberately unconstrained" -- not the same as leaving it off.
def passthrough(value: Any) -> Any:
    return value

print(apply_twice(lambda n: n + 3, 10))      # 16
print(apply_twice.__annotations__["func"])

`Callable` matters for Sections 8&ndash;10, where functions are passed around as values.

### Annotating our own pipeline

Here is the vocabulary applied to real functions over the coffee-cart table. Note that the
annotations alone describe the whole data flow.

In [ ]:
def prices_for(rows: list[dict], category: str) -> list[float]:
    """Return the prices of rows in one category."""
    return [row["price"] for row in rows if row["category"] == category]


def mean_of(values: list[float]) -> float | None:
    """Return the arithmetic mean, or None for an empty list."""
    if not values:
        return None
    return sum(values) / len(values)


drink_prices = prices_for(sales, "drink")
print(drink_prices, "->", mean_of(drink_prices))
print(prices_for(sales, "sandwich"), "->", mean_of(prices_for(sales, "sandwich")))

In [ ]:
import inspect

# `inspect.signature` reads the whole contract back out of the function object.
for func in (prices_for, mean_of, find_price):
    print(f"{func.__name__}{inspect.signature(func)}")

In [ ]:
### Try it: add annotations to this function, then check them with
### `inspect.signature`. What should the return annotation be when the
### table is empty?

def items_over(rows, threshold):
    return [row["item"] for row in rows if row["price"] > threshold]

### Enter your code here ###

---

## 8. `lambda`: Functions as Expressions

A `lambda` creates a function **without a `def` statement and without a name**. The two
definitions below produce equivalent function objects:

```python
def add(a, b):
    return a + b

add = lambda a, b: a + b
```

The syntax is:

```text
lambda parameters: single_expression
        --------  -----------------
            |             |
      same rules as   evaluated and
      a def's list    returned automatically
```

Note what is missing: no `def`, no name, no parentheses around parameters, and **no
`return`** &mdash; the expression's value *is* the return value.

In [ ]:
add = lambda a, b: a + b
print(add(2, 3))
print(type(add), add.__name__)      # a real function object, named '<lambda>'

# Defaults, *args and keyword arguments all work as usual:
greet = lambda name, greeting="Hello": f"{greeting}, {name}"
print(greet("Amina"))
print(greet("Bo", greeting="Welcome"))

### The one hard limit: a single expression

A lambda body must be **one expression**. Statements &mdash; assignment, `if`/`else` blocks,
`for`, `while`, `try`, `raise`, `return` &mdash; are not allowed.

In [ ]:
for source in [
    "lambda n: total = n + 1",        # assignment is a statement
    "lambda n: if n > 0: n",          # if-block is a statement
    "lambda n: return n",             # return is a statement
    "lambda n: raise ValueError(n)",  # raise is a statement
]:
    try:
        compile(source, "<demo>", "eval")
    except SyntaxError:
        print("SyntaxError:", source)

The **conditional expression** `a if condition else b` *is* an expression, so it is
allowed &mdash; and it covers most of what you would want an `if` for:

In [ ]:
label = lambda price: "cheap" if price < 2.00 else "not cheap"
print(label(1.75), "|", label(3.50))

### Where lambdas genuinely help

A lambda earns its place when a function is needed **as an argument**, is **small**, and is
**used once**. The canonical case is the `key=` parameter of `sorted`, `min`, and `max`.

In [ ]:
# Sort the table by price. `key` receives one row and returns the value to sort on.
by_price = sorted(sales, key=lambda row: row["price"])
for row in by_price:
    print(f"  {row['item']:<8}{row['price']:>6.2f}")

print()
print("cheapest:", min(sales, key=lambda row: row["price"])["item"])
print("priciest:", max(sales, key=lambda row: row["price"])["item"])

In [ ]:
# Sorting on two columns at once: return a tuple from the key function.
for row in sorted(sales, key=lambda row: (row["category"], -row["price"])):
    print(f"  {row['category']:<7}{row['item']:<8}{row['price']:>6.2f}")

### When *not* to use a lambda

PEP 8 is explicit: **do not assign a lambda to a name.** If it deserves a name, it deserves
a `def`, which gives you a real `__name__`, a docstring, and a better traceback.

```python
label = lambda price: ...      # discouraged
def label(price): ...          # preferred
```

In [ ]:
# Why the style rule exists: compare what the traceback can tell you.
import traceback

lambda_version = lambda n: 1 / n        # assigned a name, but stays anonymous

def def_version(n):
    return 1 / n

for func in (lambda_version, def_version):
    try:
        func(0)
    except ZeroDivisionError as error:
        innermost = traceback.extract_tb(error.__traceback__)[-1]
        print(f"called as {func.__name__:<14} -> traceback blames: {innermost.name!r}")

The `def` version names itself in the traceback. In a large program, a stack full of
`<lambda>` frames tells you nothing about where you are.

**Summary:** reach for a lambda when it is an *argument*; reach for `def` when it is a
*definition*.

In [ ]:
### Try it: use sorted() with a lambda to order `sales` by day, then by item
### name within each day. Then rewrite the same key as a named def and
### compare readability.

### Enter your code here ###

---

## 9. Functions Are Objects

In Python, a function is an **object** like any other. It has a type, it has attributes,
and a `def` statement simply **binds that object to a name** &mdash; exactly the way `x = 5`
binds an integer.

This one fact explains everything in the rest of the notebook.

In [ ]:
def square(n):
    """Return n squared."""
    return n * n

print("type    :", type(square))
print("name    :", square.__name__)
print("doc     :", square.__doc__)
print("module  :", square.__module__)
print("callable:", callable(square))

### A function can be assigned to another name

`square` is a *name*, not the function itself. Bind a second name to the same object and
both work &mdash; note there are no parentheses on the right-hand side.

In [ ]:
alias = square              # NOT square() -- that would call it and store the result
print(alias(5))
print("same object?", alias is square)
print("but __name__ still reports:", alias.__name__)

called = square(5)          # WITH parentheses: stores the RESULT, an int
print(called, type(called))

`square` versus `square()` is the distinction to hold on to: **the name is the function;
the parentheses are the call.**

### Functions can live in lists and dictionaries

In [ ]:
def double(n): return n * 2
def negate(n): return -n

# A list of functions:
for func in [square, double, negate]:
    print(f"{func.__name__:<8}(7) = {func(7)}")

In [ ]:
# A dictionary of functions -- a "dispatch table". This is how you replace
# a long if/elif chain that chooses between behaviors.

OPERATIONS = {
    "square": square,
    "double": double,
    "negate": negate,
}

def apply_operation(name: str, value: int) -> int:
    """Look up an operation by name and apply it."""
    if name not in OPERATIONS:
        raise ValueError(f"unknown operation {name!r}; expected one of {sorted(OPERATIONS)}")
    return OPERATIONS[name](value)

print(apply_operation("double", 21))
print(apply_operation("square", 9))

try:
    apply_operation("cube", 3)
except ValueError as error:
    print(f"ValueError: {error}")

Adding a fourth operation now means adding **one dictionary entry**, not editing a
conditional. That is the practical payoff of functions being ordinary values.

### Functions can be passed as arguments

A function that takes another function as a parameter is called a **higher-order
function**. You have already used several: `sorted(key=...)`, `min`, `max`.

In [ ]:
from typing import Callable

def apply_to_column(rows: list[dict], column: str,
                    func: Callable[[float], float]) -> list[float]:
    """Apply func to one column of the table and return the results."""
    return [func(row[column]) for row in rows]


def add_tax(price: float) -> float:
    return round(price * 1.08, 2)

print(apply_to_column(sales, "price", add_tax))
print(apply_to_column(sales, "price", lambda p: p * 2))     # a lambda works too
print(apply_to_column(sales, "price", round))               # a BUILT-IN works too

### The built-in higher-order functions

`map` and `filter` apply a function across a collection. Both return lazy iterators, so
wrap them in `list()` to see the values.

In [ ]:
prices = [row["price"] for row in sales]

print("map   :", list(map(add_tax, prices)))
print("filter:", list(filter(lambda p: p > 3.00, prices)))

# A comprehension usually reads better and is the Pythonic default:
print("comp  :", [add_tax(p) for p in prices])
print("comp  :", [p for p in prices if p > 3.00])

Prefer comprehensions for readability. Know `map` and `filter` because you will read them
in other people's code &mdash; and because `map` accepts any callable, which makes it handy
when you already *have* a function object.

### Functions can have attributes attached

In [ ]:
def compute_rmse(predicted, actual):
    return sum((p - a) ** 2 for p, a in zip(predicted, actual)) ** 0.5

compute_rmse.units = "dollars"          # functions are objects, so this just works
compute_rmse.version = 2

print(compute_rmse.units, "| version", compute_rmse.version)
print(sorted(k for k in vars(compute_rmse)))

In [ ]:
### Try it: build a dispatch table mapping "mean", "min", "max" to functions
### that take a list of prices, then write `summarize(prices, how)` that
### uses it. Raise ValueError for an unknown `how`.

### Enter your code here ###

---

## 10. Closures and Currying

If a function can be passed *in* as a value, it can also be returned *out* as one.

### A function that returns a function

In [ ]:
def make_multiplier(factor):
    """Return a NEW function that multiplies its argument by factor."""

    def multiply(value):
        return value * factor      # `factor` comes from the ENCLOSING scope

    return multiply                # returned WITHOUT parentheses -- the object itself


double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10), triple(10))
print(double, triple, sep="\n")

### What is a closure?

`make_multiplier` has returned, so its local `factor` should be gone. Yet `double` still
remembers `factor == 2`.

A function that remembers names from the scope where it was **created** is a **closure**.
Python keeps those values alive in the function object, and you can inspect them.

In [ ]:
print("free variables:", double.__code__.co_freevars)
print("captured cell :", double.__closure__)
print("captured value:", double.__closure__[0].cell_contents)
print("triple's value:", triple.__closure__[0].cell_contents)

Each call to `make_multiplier` creates a **separate** closure with its own captured value.
That is why `double` and `triple` do not interfere.

### Currying

**Currying** means turning a function of several arguments into a chain of functions that
each take one argument. The general form is a function returning a function:

In [ ]:
def power(exponent):
    """Curried power: power(2) returns a squaring function."""
    def raise_to(base):
        return base ** exponent
    return raise_to


square = power(2)
cube = power(3)

print(square(5), cube(5))
print(power(2)(5))        # call both stages at once -- note the two argument lists

`power(2)(5)` shows the mechanism plainly: `power(2)` evaluates to a function, and `(5)`
calls *that*.

A fully curried three-argument function chains three times:

In [ ]:
def curried_volume(length):
    return lambda width: lambda height: length * width * height

print(curried_volume(2)(3)(4))

# Each stage is a reusable, configured function:
base_2 = curried_volume(2)
base_2_by_3 = base_2(3)
print(base_2_by_3(4), base_2_by_3(10))

### Why this is useful: configured functions

The practical payoff is building a **specialized function once** and reusing it, instead of
repeating a configuration argument at every call.

In [ ]:
def make_tax_calculator(rate: float, label: str) -> "Callable[[float], float]":
    """Return a function that applies one fixed tax rate.

    Args:
        rate: The tax rate as a fraction, e.g. 0.08 for 8%.
        label: A name for the jurisdiction, used in the result string.

    Returns:
        A function taking a price and returning the taxed price.
    """
    def calculate(price: float) -> float:
        return round(price * (1 + rate), 2)

    calculate.__name__ = f"tax_{label}"     # functions are objects -- rename it
    calculate.__doc__ = f"Apply the {label} rate of {rate:.0%}."
    return calculate


georgia = make_tax_calculator(0.04, "georgia")
atlanta = make_tax_calculator(0.089, "atlanta")

print(georgia.__name__, "->", georgia.__doc__)
for row in sales[:3]:
    print(f"  {row['item']:<8}{row['price']:>6.2f}"
          f"{georgia(row['price']):>8.2f}{atlanta(row['price']):>8.2f}")

Because `georgia` and `atlanta` are ordinary function objects, they slot straight into the
higher-order functions from Section 9:

In [ ]:
print(apply_to_column(sales, "price", georgia))
print(apply_to_column(sales, "price", atlanta))

# ...and into a dispatch table:
JURISDICTIONS = {"GA": georgia, "ATL": atlanta}
print(JURISDICTIONS["ATL"](3.50))

### `functools.partial`: currying from the standard library

When you just want to **fix some arguments** of an existing function, you do not need to
write a closure by hand. `functools.partial` does it.

In [ ]:
from functools import partial

def apply_tax(price: float, rate: float) -> float:
    """Return price with tax applied. Two arguments, ordinary function."""
    return round(price * (1 + rate), 2)


# Fix `rate` and get back a one-argument function:
georgia_partial = partial(apply_tax, rate=0.04)

print(georgia_partial(3.50))
print(georgia_partial(2.75))
print(georgia_partial)                 # partial objects show what they captured
print("wrapped function:", georgia_partial.func.__name__)
print("fixed keywords  :", georgia_partial.keywords)

Use `partial` when the function already exists; write a closure by hand when you want to
compute something at configuration time, attach metadata, or return something more
elaborate.

### The late-binding trap

A closure captures the **variable**, not the value it had at capture time. Creating
closures in a loop is the classic way to get this wrong.

In [ ]:
# WRONG: all three closures share the same `factor` variable.
multipliers = []
for factor in [2, 3, 4]:
    multipliers.append(lambda value: value * factor)

print([m(10) for m in multipliers])     # [40, 40, 40] -- factor ended at 4


# RIGHT: bind the current value as a default argument, evaluated at definition time.
multipliers = []
for factor in [2, 3, 4]:
    multipliers.append(lambda value, factor=factor: value * factor)

print([m(10) for m in multipliers])     # [20, 30, 40]


# ALSO RIGHT: a factory function gives each closure its own scope.
multipliers = [make_multiplier(factor) for factor in [2, 3, 4]]
print([m(10) for m in multipliers])

In [ ]:
### Try it: write `make_column_getter(column)` that returns a function taking
### one row and returning that column's value. Use it as the `key=` argument
### to sorted() -- e.g. `sorted(sales, key=make_column_getter("price"))`.
###
### Then compare your version to the standard library's `operator.itemgetter`.

### Enter your code here ###

---

## Summary

### The parts of a function

| Part | Syntax | Notes |
|---|---|---|
| Definition | `def name(params) -> T:` | Executed at definition time; body runs at call time |
| Positional parameter | `def f(a)` | Matched by position |
| Default | `def f(a=1)` | Makes it optional; must follow non-default parameters |
| Keyword argument | `f(a=1)` | Matched by name; order free |
| Variadic positional | `def f(*args)` | Collects extras into a **tuple** |
| Variadic keyword | `def f(**kwargs)` | Collects extras into a **dict** |
| Positional-only | `def f(a, /)` | Cannot be passed by name |
| Keyword-only | `def f(*, a)` | Cannot be passed by position |
| Unpacking a call | `f(*seq)`, `f(**mapping)` | Spreads a collection into arguments |
| Return | `return value` | Ends the call; default result is `None` |
| Annotation | `a: int -> str` | Documentation and tooling; **not enforced** |
| Lambda | `lambda a: expr` | One expression, no name, no `return` |

### The ideas worth remembering

1. **A `def` binds a function object to a name.** `square` is the object; `square()` is the
   call.
2. **Names resolve by LEGB** &mdash; local, enclosing, global, built-in &mdash; and assigning
   to a name anywhere in a body makes it local everywhere in that body.
3. **Arguments are passed by assignment.** Rebinding a parameter is invisible to the
   caller; mutating the object is not.
4. **Pass by name when the value alone is not self-explanatory**, and never use a mutable
   default.
5. **Annotations are for readers and tools, not the runtime.** Validate at boundaries.
6. **Use a lambda as an argument, a `def` as a definition.**
7. **Functions are objects**, so they go in lists, dictionaries, parameters, and return
   values &mdash; which is what makes dispatch tables, `key=` functions, closures, and
   currying possible.

### Where to go next

- *Deep Dive: Why Functions Are Designed This Way* &mdash; the design reasoning behind these
  mechanics: contracts, side effects, purity, and testing.
- Reference: [PEP 8 &mdash; Style Guide](https://peps.python.org/pep-0008/),
  [PEP 257 &mdash; Docstring Conventions](https://peps.python.org/pep-0257/),
  [PEP 484 &mdash; Type Hints](https://peps.python.org/pep-0484/).